# Currency & Economy

Currency flow simulation for Animal Company. Models instance value-in/out, wallet balances, Key Card progression, and Battle Pass economics.

> Reference: `animalco_economy.md` in brain-rag vault

In [1]:
from IPython.display import HTML
display(HTML('''
<style id="aco-toggle-style"></style>
<button id="aco-toggle-btn"
  onclick="
    var style = document.getElementById('aco-toggle-style');
    if (style.innerHTML === '') {
      style.innerHTML = '.jp-Cell-inputWrapper { display: none !important; }';
      this.innerHTML = 'Show Code';
    } else {
      style.innerHTML = '';
      this.innerHTML = 'Hide Code';
    }
  "
  style="padding: 6px 16px; font-size: 13px; cursor: pointer;
         border: 1px solid #ccc; border-radius: 4px; background: #f5f5f5;">
  Hide Code
</button>
'''))

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

from aco_model.models import EconomyParams, RetentionCurve
from aco_model.retention import load_installs, simulate
from aco_model.economy import simulate_economy
from aco_model.config import load_config
from aco_model.state import load_state, save_state, DEFAULT_STATE_PATH

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

_HEADLESS = os.environ.get('ACO_HEADLESS') == '1'

cfg = load_config()
state = load_state()

if state is not None:
    installs = load_installs(cfg.installs_path)
    curve = RetentionCurve(anchors=state.retention_anchors)
    sim = simulate(installs, curve, state.sim_days)
    econ_params = state.economy or cfg.economy
    print(f'=== Loaded from shared state ===')
else:
    installs = load_installs(cfg.installs_path)
    sim = simulate(installs, cfg.retention, cfg.sim_days)
    econ_params = cfg.economy
    print(f'=== Using config.yaml defaults ===')

print(f'  DAU range: {sim.dau.min():,} – {sim.dau.max():,}')
print(f'  Instances/day: {econ_params.instances_per_day}')

## 1. Instance Economics

In [ ]:
runs_slider = widgets.IntSlider(value=econ_params.instances_per_day, min=1, max=10, step=1,
                                 description='Runs/day:')
buff_cost_slider = widgets.FloatSlider(value=econ_params.buff_cost_scrap, min=0, max=200, step=10,
                                       description='Buff cost:')
bp_rate_slider = widgets.FloatSlider(value=econ_params.battle_pass.purchase_rate * 100,
                                      min=1, max=30, step=1, description='BP buyers %:')

_anchors = state.retention_anchors if state else cfg.retention.anchors
_monetization = state.monetization if state else cfg.monetization

def _make_econ_params(runs, buff_cost, bp_rate):
    bp = econ_params.battle_pass.model_copy(update={'purchase_rate': bp_rate / 100.0})
    return econ_params.model_copy(update={
        'instances_per_day': runs, 'buff_cost_scrap': buff_cost, 'battle_pass': bp,
    })

# Output widgets for each section
out_ie = widgets.Output()
out_flows = widgets.Output()
out_wallet = widgets.Output()
out_kc = widgets.Output()
out_bp = widgets.Output()

def _update_all(*args):
    runs, buff_cost, bp_rate = runs_slider.value, buff_cost_slider.value, bp_rate_slider.value
    params = _make_econ_params(runs, buff_cost, bp_rate)
    econ = simulate_economy(sim, params)
    days = np.arange(1, sim.sim_days + 1)

    # Section 1: Instance Economics
    with out_ie:
        out_ie.clear_output(wait=True)
        ie = econ.instance_economics_dataframe()
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        x = np.arange(len(ie)); width = 0.35
        ax1.bar(x - width/2, ie['value_in'], width, label='Value In', color='#F44336', alpha=0.7)
        ax1.bar(x + width/2, ie['value_out'], width, label='Value Out', color='#4CAF50', alpha=0.7)
        ax1.set_xticks(x); ax1.set_xticklabels(ie['tier'])
        ax1.set_ylabel('Value (currency units)'); ax1.set_title('Value In vs Out per Instance Run'); ax1.legend()
        ax2.bar(ie['tier'], ie['net_value'], color='#2196F3', alpha=0.7)
        ax2.set_ylabel('Net Value'); ax2.set_title('Net Value per Instance Run (player profit)')
        ax2.axhline(y=0, color='red', linestyle='--', alpha=0.5)
        plt.tight_layout(); plt.show()
        print(ie.to_string(index=False))

    # Section 2: Currency Flows
    with out_flows:
        out_flows.clear_output(wait=True)
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        for ax, earned, spent, name in [
            (axes[0], econ.daily_nuts_earned, econ.daily_nuts_spent, 'Nuts'),
            (axes[1], econ.daily_scrap_earned, econ.daily_scrap_spent, 'Scrap'),
        ]:
            ax.fill_between(days, earned, alpha=0.3, color='#4CAF50', label='Earned')
            ax.fill_between(days, spent, alpha=0.3, color='#F44336', label='Spent')
            ax.plot(days, earned, linewidth=2, color='#4CAF50')
            ax.plot(days, spent, linewidth=2, color='#F44336')
            ax.set_xlabel('Day'); ax.set_ylabel(name); ax.set_title(f'Daily {name}: Earned vs Spent'); ax.legend()
        axes[2].plot(days, econ.daily_keycards_consumed, linewidth=2, color='#F44336', label='Consumed')
        axes[2].plot(days, econ.daily_keycards_net, linewidth=2, color='#FF9800', label='Net consumed')
        axes[2].set_xlabel('Day'); axes[2].set_ylabel('Keycards'); axes[2].set_title('Daily Keycard Consumption'); axes[2].legend()
        plt.tight_layout(); plt.show()

    # Section 3: Wallet Balances
    with out_wallet:
        out_wallet.clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        ax1.plot(days, econ.nuts_balance, linewidth=2, color='#FF9800', label='Nuts')
        ax1.plot(days, econ.scrap_balance, linewidth=2, color='#9C27B0', label='Scrap')
        ax1.set_xlabel('Day'); ax1.set_ylabel('Cumulative Balance'); ax1.set_title('Total Currency in System'); ax1.legend()
        dau = sim.dau.astype(float)
        with np.errstate(divide='ignore', invalid='ignore'):
            nuts_pp = np.where(dau > 0, econ.nuts_balance / dau, 0)
            scrap_pp = np.where(dau > 0, econ.scrap_balance / dau, 0)
        ax2.plot(days, nuts_pp, linewidth=2, color='#FF9800', label='Nuts/player')
        ax2.plot(days, scrap_pp, linewidth=2, color='#9C27B0', label='Scrap/player')
        ax2.set_xlabel('Day'); ax2.set_ylabel('Avg Balance per Player'); ax2.set_title('Avg Wallet Balance per Active Player'); ax2.legend()
        plt.tight_layout(); plt.show()
        df = econ.to_dataframe()
        print(f'Day {sim.sim_days}: Nuts balance = {df.iloc[-1]["nuts_balance"]:,} | Scrap balance = {df.iloc[-1]["scrap_balance"]:,}')

    # Section 4: Key Card Progression
    with out_kc:
        out_kc.clear_output(wait=True)
        kc = econ.keycard_progression_dataframe()
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        ax1.bar(kc['tier'], kc['merge_cost_nuts'], color='#FF9800', alpha=0.7)
        ax1.set_ylabel('Nuts'); ax1.set_title('Merge Cost per Tier')
        ax2.bar(kc['tier'], kc['cumulative_nuts'], color='#F44336', alpha=0.7, label='Cumulative nuts')
        ax2_r = ax2.twinx()
        ax2_r.plot(kc['tier'], kc['cumulative_cards'], 'o-', color='#2196F3', linewidth=2, label='Cumulative cards')
        ax2.set_ylabel('Nuts', color='#F44336'); ax2_r.set_ylabel('Cards', color='#2196F3')
        ax2.set_title('Cumulative Investment to Reach Tier')
        ax2.legend(loc='upper left'); ax2_r.legend(loc='upper right')
        plt.tight_layout(); plt.show()
        avg_nuts_per_run = np.mean([t.nuts_earned for t in params.instance_tiers])
        nuts_per_day = avg_nuts_per_run * params.instances_per_day
        print(f'Avg nuts earned per day (per player): {nuts_per_day:,.0f}')
        for _, row in kc.iterrows():
            if row['cumulative_nuts'] > 0:
                print(f'  {row["tier"]}: ~{row["cumulative_nuts"] / nuts_per_day:.1f} days to afford')

    # Section 5: Battle Pass
    with out_bp:
        out_bp.clear_output(wait=True)
        bp = params.battle_pass
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        cum_bp_rev = np.cumsum(econ.battle_pass_daily_revenue)
        ax1.plot(days, cum_bp_rev, linewidth=2, color='#4CAF50')
        ax1.fill_between(days, cum_bp_rev, alpha=0.2, color='#4CAF50')
        ax1.set_xlabel('Day'); ax1.set_ylabel('Cumulative BP Revenue (USD)'); ax1.set_title('Battle Pass Revenue')
        rates = [0.1, 0.2, 0.3, 0.5, 0.7]
        for rate in rates:
            bp_var = bp.model_copy(update={'completion_rate': rate})
            params_var = params.model_copy(update={'battle_pass': bp_var})
            econ_var = simulate_economy(sim, params_var)
            net_coins = np.cumsum(econ_var.daily_coins_from_bp - econ_var.daily_coins_returned_bp)
            style = {'linewidth': 3, 'linestyle': '--'} if rate == bp.completion_rate else {'linewidth': 2}
            ax2.plot(days, net_coins * params.coin_to_usd, label=f'{rate:.0%} complete', **style)
        ax2.set_xlabel('Day'); ax2.set_ylabel('Net Coin Revenue (USD)')
        ax2.set_title('BP Net Revenue by Completion Rate'); ax2.legend()
        plt.tight_layout(); plt.show()
        print(f'BP: {bp.cost_coins} coins (${bp.cost_coins * params.coin_to_usd:.2f}), {bp.season_days}-day season')
        print(f'Total BP revenue: ${econ.battle_pass_total_revenue:,.2f}')

    # Save state
    save_state(sim, _anchors, monetization=_monetization, economy=params)

if _HEADLESS:
    _update_all()
else:
    for s in [runs_slider, buff_cost_slider, bp_rate_slider]:
        s.observe(lambda change: _update_all(), names='value')
    display(widgets.VBox([widgets.HBox([runs_slider, buff_cost_slider, bp_rate_slider]), out_ie]))
    _update_all()

## 2. Currency Flows

In [ ]:
if not os.environ.get('ACO_HEADLESS'):
    display(out_flows)

## 3. Wallet Balances

Cumulative currency in the system over time (per GameGou call recommendations).

In [ ]:
if not os.environ.get('ACO_HEADLESS'):
    display(out_wallet)

## 4. Key Card Progression

In [ ]:
if not os.environ.get('ACO_HEADLESS'):
    display(out_kc)

## 5. Battle Pass Economics

In [ ]:
if not os.environ.get('ACO_HEADLESS'):
    display(out_bp)